# ☀️ Solar Radiation Forecasting Pipeline

### 🎯 Objective:
Build a machine learning model to predict next-day solar radiation.

**Data: Open-Meteo**

In [18]:
import base64; print(base64.b64encode(open('sample_data/open_meteo.csv','rb').read()).decode())

...


In [19]:
import pandas as pd
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error

## Step 1 & 2: Data Loading & Cleaning
We use `skiprows=2` to bypass the header junk so Pandas correctly identifies our true column headers (`time`, `temperature_2m_mean`, `shortwave_radiation_sum`, etc.).

In [20]:
om_df = pd.read_csv('sample_data/open_meteo.csv', skiprows=2, on_bad_lines='skip')

print("✅ New Dataset shape:", om_df.shape)
print("✅ New Columns found:", om_df.columns.tolist())
display(om_df.head(3))
display(om_df.tail(3))

✅ New Dataset shape: (681, 6)
✅ New Columns found: ['time', 'temperature_2m_mean (°C)', 'shortwave_radiation_sum (MJ/m²)', 'sunshine_duration (s)', 'weather_code (wmo code)', 'precipitation_sum (mm)']


,time,temperature_2m_mean (°C),shortwave_radiation_sum (MJ/m²),sunshine_duration (s),weather_code (wmo code),precipitation_sum (mm)
0,2025-01-01,26.6,19.20,41870.54,61,3.3
1,2025-01-02,26.8,19.63,39174.38,63,7.2
2,2025-01-03,26.8,21.30,42508.74,63,6.6


,time,temperature_2m_mean (°C),shortwave_radiation_sum (MJ/m²),sunshine_duration (s),weather_code (wmo code),precipitation_sum (mm)
678,2026-11-10,26.0,16.61,31580.67,65,21.0
679,2026-11-11,26.2,17.00,25200.00,65,10.5
680,2026-11-12,27.4,18.48,41567.88,55,2.0


## Step 3: Engineering the Target Variable (`shift(-1)`)
* **Why we do this:** To predict the *future* (tomorrow's solar radiation), our model needs to look at today's weather inputs and connect them to tomorrow's output.
* **The Technique:** We use Pandas `.shift(-1)` to slide tomorrow's radiation value up into today's row. This aligns today's weather features with tomorrow's actual solar outcome.

In [21]:
# 3. Prepare Date and Target Variable
om_df['time'] = pd.to_datetime(om_df['time'])
om_df = om_df.sort_values('time').reset_index(drop=True)

# Define our radiation column name
rad_col = 'shortwave_radiation_sum (MJ/m²)'

# Shift radiation by -1 so each day's row contains tomorrow's radiation value
om_df["target"] = om_df[rad_col].shift(-1)

# Drop the last row since we don't have tomorrow's data for the final day
om_df = om_df.dropna(subset=["target"])

# print("✅ Target variable created successfully!")
# print("Total usable rows after shift:", len(om_df))
# print(om_df[['time', rad_col, 'target']].head(3))
display(om_df.head(3))
display(om_df.tail(3))

,time,temperature_2m_mean (°C),shortwave_radiation_sum (MJ/m²),sunshine_duration (s),weather_code (wmo code),precipitation_sum (mm),target
0,2025-01-01,26.6,19.20,41870.54,61,3.3,19.63
1,2025-01-02,26.8,19.63,39174.38,63,7.2,21.30
2,2025-01-03,26.8,21.30,42508.74,63,6.6,19.30


,time,temperature_2m_mean (°C),shortwave_radiation_sum (MJ/m²),sunshine_duration (s),weather_code (wmo code),precipitation_sum (mm),target
677,2026-11-09,25.4,13.32,21708.73,65,28.9,16.61
678,2026-11-10,26.0,16.61,31580.67,65,21.0,17.00
679,2026-11-11,26.2,17.00,25200.00,65,10.5,18.48


## Step 4: Chronological Train/Test Split
* **Why chronological (not random)?** In time-series forecasting, random splitting allows the model to "cheat" by training on future data to predict the past.
* **Our Strategy:** We split the data chronologically—training on the first 80% of days and testing on the final 20% (the most recent data) to simulate a realistic, real-world deployment scenario.

In [22]:
# 1. Chronological Train/Test Split (80% train, 20% test)
train_size = int(len(om_df) * 0.8)
train_df = om_df.iloc[:train_size]
test_df = om_df.iloc[train_size:]

# print(f"📊 Training set rows: {len(train_df)}")
# print(f"📊 Testing set rows: {len(test_df)}")

## Step 5: Baseline Comparison & Model Training
* **The Baseline ("Persistence Model"):** We assume tomorrow's weather will be identical to today's. This gives us a benchmark error score (MAE) that our AI model *must* beat to prove it learned something useful.
* **Linear Regression:** We train a linear model on key weather features (radiation, sunshine duration, temperature, and precipitation) to see if mathematical patterns can outperform the naive baseline.

In [ ]:
# 1. Select features from our clean dataset
features = [
    'shortwave_radiation_sum (MJ/m²)',
    'sunshine_duration (s)',
    'temperature_2m_mean (°C)',
    'precipitation_sum (mm)'
]
# Ensure we only use columns that actually exist in the dataframe
features = [f for f in features if f in om_df.columns]

X_train = train_df[features]
y_train = train_df['target']
X_test = test_df[features]
y_test = test_df['target']

# 2. Calculate Baseline ("Tomorrow equals today")
baseline_preds = test_df['shortwave_radiation_sum (MJ/m²)']
baseline_mae = mean_absolute_error(y_test, baseline_preds)
print(f"📉 Baseline MAE: {baseline_mae:.4f}")

# 3. Train Linear Regression Model
model = LinearRegression()
model.fit(X_train, y_train)

model_preds = model.predict(X_test)
model_mae = mean_absolute_error(y_test, model_preds)
print(f"📈 Linear Regression MAE: {model_mae:.4f}")

# 4. Check if we won!
if model_mae < baseline_mae:
    print("\n🎉 BOOM! Your model successfully beat the baseline!")
else:
    print("\n💪 Model trained! Ready to tune features.")

📉 Baseline MAE: 2.3782
📈 Linear Regression MAE: 2.2273

🎉 BOOM! Your model successfully beat the baseline!
